# Perfil económico de los municipios de Morelos por tipo de actividad

> ## Pregunta de investigación
>
> **¿Qué municipios del estado de Morelos comparten un perfil económico similar según la concentración
> de sus unidades económicas dedicadas a la manufactura, el comercio y los servicios?**

**Tema:** 2 — Perfil económico de municipios por tipo de actividad
**Modelo:** KMeans (aprendizaje **no supervisado**)
**Unidad de análisis:** el municipio, no el establecimiento individual.

---

## 0.1 Justificación: por qué esta pregunta

**1. Responde a lo que el agrupamiento sabe hacer.** El objetivo es identificar municipios con perfiles
económicos afines. KMeans permite descubrir **"vecindarios económicos"**: grupos de municipios con una
estructura de unidades económicas parecida **independientemente de su cercanía geográfica**. La geografía
no entra al modelo — si al final aparece un patrón territorial, será un hallazgo, no un supuesto
(lo verificamos en la gráfica 6).

**2. Descubre patrones que la estadística descriptiva no alcanza.** Comparar 36 municipios a lo largo de
9 variables al mismo tiempo no es algo que se pueda hacer leyendo tablas. El aprendizaje no supervisado
permite una categorización basada en datos, capaz de aislar —por ejemplo— demarcaciones de vocación
netamente comercial frente a otras con un perfil industrial consolidado.

**3. Tiene utilidad práctica.** Conocer la composición de unidades económicas de una región convierte
datos crudos en información útil: focalización de servicios B2B, distribución de soluciones
especializadas o diseño de política pública de desarrollo económico regional.

**4. Y tiene un alcance acotado, que declaramos desde el inicio.** La segmentación se basa
**exclusivamente en la frecuencia y el volumen de unidades económicas registradas formalmente**. No
contempla facturación, márgenes de ingreso ni el sector informal, porque el DENUE simplemente no los
captura. Las conclusiones se limitan a la **composición estructural del mercado formal** — esto se
desarrolla en la sección 16.

## 0. Procedencia de los datos

| Campo | Valor |
|---|---|
| **Fuente** | DENUE — Directorio Estadístico Nacional de Unidades Económicas |
| **Institución** | INEGI (Instituto Nacional de Estadística y Geografía) |
| **URL de descarga** | https://www.inegi.org.mx/app/descarga/?ti=6 |
| **Corte / versión** | **DENUE 2026/06** (reporte de junio de 2026) |
| **Fecha de descarga** | Agosto de 2026 (el archivo se incorporó al repositorio el 2026-08-23) |
| **Cobertura geográfica** | Estado de Morelos (`cve_ent = 17`), 36 municipios |
| **Unidad de registro** | Un establecimiento (unidad económica) por fila |
| **Registros** | 113,066 |
| **Variables originales** | 42 |
| **Codificación del archivo** | `latin-1` (el DENUE no se distribuye en UTF-8) |

**¿Por qué consideramos confiable la fuente?**
El DENUE es el directorio oficial de establecimientos del INEGI, construido a partir de los Censos
Económicos y actualizado con recorridos de campo. Es de acceso público, gratuito y descargable en CSV,
lo que hace el análisis reproducible por cualquier persona.

**Limitación desde el origen:** el DENUE registra la **existencia** de un establecimiento y su actividad,
pero **no** su producción, ingresos ni valor agregado. Esto acota lo que podemos afirmar y se retoma en
la sección 16.

---
## 1. Carga de librerías y del dataset

In [ ]:
# Librerías de manejo de datos
import pandas as pd
import numpy as np
# Gráficas
import matplotlib.pyplot as plt
# Manejo de rutas
from pathlib import Path

# Modelo no supervisado y herramientas de apoyo
from sklearn.preprocessing import StandardScaler   # escalado
from sklearn.cluster import KMeans                 # modelo
from sklearn.metrics import silhouette_score       # métrica
from sklearn.decomposition import PCA              # graficar en 2D

# Semilla global para replicabilidad
np.random.seed(42)

# Consistencia del formato
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
# Versiones utilizadas:
import sys
print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
import matplotlib, sklearn
print('matplotlib:', matplotlib.__version__)
print('scikit-learn:', sklearn.__version__)

In [ ]:
# Derivamos la ruta base para evitar que fallen las rutas relativas
BASE = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()

RUTA_RAW = BASE / 'data' / 'raw' / 'dataset_original.csv'
RUTA_PROCESSED = BASE / 'data' / 'processed'
RUTA_OUTPUTS = BASE / 'outputs'
RUTA_IMAGENES = BASE / 'imagenes'
# Validar existencia del dataset
assert RUTA_RAW.exists(), f'No se encontró el dataset en {RUTA_RAW}'

print('Carpeta base del proyecto:', BASE.resolve())
print('Dataset original:', RUTA_RAW.name, '(', round(RUTA_RAW.stat().st_size / 1024 / 1024, 1), 'MB )')

In [ ]:
# encoding='latin-1' porque el DENUE no viene en UTF-8
# low_memory=False evita el aviso de tipos mixtos en un archivo de 113 mil filas.
df = pd.read_csv(RUTA_RAW, encoding='latin-1', low_memory=False)
print('Dataset cargado.')

In [ ]:
df.head()

---
## 2. Revisión inicial del dataset

Antes de tocar nada, queremos saber con qué estamos trabajando: cuántas filas y columnas hay,
qué tipos de datos tiene y cómo se ven las columnas numéricas.

In [ ]:
# Dimensiones del dataset (filas, columnas)
print('Shape del dataset:', df.shape)

In [ ]:
# Nombres de todas las columnas
print('Columnas del dataset:')
print(list(df.columns))

In [ ]:
# Tipos de datos por columna
df.dtypes

In [ ]:
# Información general: tipos, no nulos y memoria usada
df.info()

In [ ]:
# Estadísticas descriptivas de las columnas numéricas.
df.describe()

---
## 3. Revisión de duplicados

Un establecimiento repetido inflaría artificialmente el conteo de un municipio y por lo tanto
su perfil económico. Lo revisamos antes de cualquier otra cosa.

In [ ]:
# Contar filas duplicadas
duplicados = df.duplicated().sum()
print('Filas duplicadas:', duplicados)

In [ ]:
# Solo eliminamos si realmente hay duplicados, if para dejar documentado si se eliminó algo
if duplicados > 0:
    df = df.drop_duplicates()
    print('Duplicados eliminados. Nuevo shape:', df.shape)
else:
    print('No se encontraron duplicados. No se requiere acción.')

---
## 4. Revisión de valores nulos

In [ ]:
# Total de nulos por columna
nulos = df.isnull().sum()
print('Nulos por columna:')
print(nulos)

In [ ]:
# Porcentaje de nulos, mostrando solo las columnas que sí tienen nulos.
# Nos interesa el porcentaje y no el conteo para dimensionar el problema.
porcentaje_nulos = (df.isnull().sum() / len(df) * 100).round(2)
nulos_filtrados = porcentaje_nulos[porcentaje_nulos > 0].sort_values(ascending=False)
print('Porcentaje de nulos (solo columnas con nulos):')
print(nulos_filtrados)

**Interpretación:** Las columnas con porcentajes altos de nulos es debido a que son opcionales

Lo importante es que las columnas que sí vamos a usar (`municipio`, `nombre_act`, `codigo_act`,
`per_ocu`, `tipoUniEco`, `latitud`, `longitud`) no tienen ningún nulo. Por eso la decisión de limpieza
no es imputar ni rellenar, es descartar las columnas que no aportan al análisis

---
## 5. Revisión de valores únicos en columnas clave

Aquí verificamos que los datos digan lo que creemos que dicen: que sea solo Morelos, que estén los
36 municipios y qué forma tienen las variables categóricas que vamos a usar.

In [ ]:
# Verificar que el archivo descargado contiene únicamente Morelos
print('Entidades únicas:', df['entidad'].nunique())
print(df['entidad'].unique())
print('Clave de entidad:', df['cve_ent'].unique())

In [ ]:
# Municipios: deben ser los 36 de Morelos
print('Municipios únicos:', df['municipio'].nunique())
print(sorted(df['municipio'].unique()))

In [ ]:
# Actividades económicas más frecuentes.
print('Top 10 actividades económicas:')
print(df['nombre_act'].value_counts().head(10))

In [ ]:
# per_ocu no es un número de empleados, es un rango categórico.
print('Distribución de personal ocupado (per_ocu):')
print(df['per_ocu'].value_counts())

In [ ]:
# tipoUniEco distingue establecimientos fijos de semifijos (puestos, tianguis).
print('Tipos de unidad económica:')
print(df['tipoUniEco'].value_counts())

In [ ]:
# Corte temporal: fecha_alta indica cuándo se dio de alta el registro.
print('fecha_alta más antigua :', df['fecha_alta'].min())
print('fecha_alta más reciente:', df['fecha_alta'].max())
print()
print('Mayores altas por año:')
print(df['fecha_alta'].astype(str).str[:4].value_counts().head(5))

In [ ]:
# Validar que el corte de datos sea como se documentó, revisando que no hayan fechas mayores a Junio del 2026
CORTE_DENUE = '2026-06'

alta_maxima = df['fecha_alta'].max()
print('Corte declarado del DENUE :', CORTE_DENUE)
print('Alta más reciente del archivo:', alta_maxima)
print()

if alta_maxima <= CORTE_DENUE:
    print('OK: el archivo es consistente con el corte declarado')
    print('No hay registros posteriores a', CORTE_DENUE)
else:
    print('ATENCIÓN: hay altas posteriores al corte declarado. Revisar la versión.')

**Hallazgos de la revisión inicial:**

1. El archivo contiene una sola entidad, Morelos, y los 36 municipios completos.
2. `per_ocu` es un rango categórico, no un conteo. El 90% de los registros cae en `0 a 5 personas`.
3. `tipoUniEco` tiene solo dos valores: `Fijo` (110,174) y `Semifijo` (2,892).
4. El comercio al por menor domina el directorio: las actividades más frecuentes son abarrotes,
   ropa y alimentos preparados. Esto anticipa que la diferencia entre municipios no va a estar en
   quién tiene comercio, sino en el margen: quién tiene además manufactura, turismo o servicios

---
## 6. Limpieza básica

1. **Seleccionar** solo las columnas relevantes (de 42 a 11): descartamos domicilio, contacto y
   claves geográficas finas que no aportan al perfil económico municipal.
2. **Crear `sector_nombre`**: el sector SCIAN a partir de los 2 primeros dígitos de `codigo_act`.
   Sin esto solo tendríamos ~1,000 actividades sueltas, imposibles de comparar entre municipios.
3. **Crear `cve_geo`**: la clave INEGI estándar de 5 dígitos (entidad + municipio), para trazabilidad.
4. **Verificar** que no queden nulos ni identificadores repetidos en las columnas que sí usamos.

In [ ]:
# Seleccionamos las columnas relevantes para el perfil económico municipal.
columnas_relevantes = [
    'id',           # Identificador único de la unidad económica
    'codigo_act',   # Código de actividad económica (SCIAN)
    'nombre_act',   # Nombre de la actividad económica
    'per_ocu',      # Rango de personal ocupado
    'cve_ent',      # Clave de la entidad (17 = Morelos)
    'cve_mun',      # Clave del municipio
    'municipio',    # Nombre del municipio
    'entidad',      # Nombre de la entidad
    'tipoUniEco',   # Tipo de unidad económica (Fijo / Semifijo)
    'latitud',      # Latitud del establecimiento
    'longitud'      # Longitud del establecimiento
]

df_sel = df[columnas_relevantes].copy()
print('Shape después de seleccionar columnas:', df_sel.shape)
df_sel.head()

In [ ]:
# El SCIAN es jerárquico: los 2 primeros dígitos del código de actividad
# identifican el SECTOR económico. Así pasamos de 1,000 actividades
# distintas a 20 sectores comparables entre municipios.
df_sel['sector_codigo'] = df_sel['codigo_act'].astype(str).str[:2]

# Mapeo oficial de los sectores SCIAN presentes en el archivo.
# Manufactura ocupa tres códigos (31, 32, 33) por diseño del propio SCIAN,
# igual que transporte (48, 49): los unificamos con el mismo nombre.
sectores_scian = {
    '11': 'Agricultura y ganadería',
    '21': 'Minería',
    '22': 'Electricidad y agua',
    '23': 'Construcción',
    '31': 'Manufactura',
    '32': 'Manufactura',
    '33': 'Manufactura',
    '43': 'Comercio al por mayor',
    '46': 'Comercio al por menor',
    '48': 'Transporte',
    '49': 'Transporte',
    '51': 'Información y medios',
    '52': 'Servicios financieros',
    '53': 'Servicios inmobiliarios',
    '54': 'Servicios profesionales',
    '55': 'Corporativos',
    '56': 'Servicios de apoyo',
    '61': 'Servicios educativos',
    '62': 'Servicios de salud',
    '71': 'Esparcimiento y cultura',
    '72': 'Alojamiento y alimentos',
    '81': 'Otros servicios',
    '93': 'Gobierno'
}

df_sel['sector_nombre'] = df_sel['sector_codigo'].map(sectores_scian)
print('Unidades económicas por sector SCIAN:')
print(df_sel['sector_nombre'].value_counts())

In [ ]:
# Verificamos que el mapeo haya cubierto todos los códigos.
sin_sector = df_sel['sector_nombre'].isnull().sum()
print('Unidades sin sector asignado:', sin_sector)

if sin_sector > 0:
    print('Códigos de sector no mapeados:')
    print(df_sel[df_sel['sector_nombre'].isnull()]['sector_codigo'].unique())

In [ ]:
# Si hubiera unidades sin sector las eliminaríamos, sin sector no pueden
# entrar en la composición sectorial del municipio.
if sin_sector > 0:
    df_sel = df_sel.dropna(subset=['sector_nombre'])
    print('Filas eliminadas sin sector. Nuevo shape:', df_sel.shape)
else:
    print('Todas las filas tienen sector asignado. No se requiere acción.')

In [ ]:
# cve_geo: clave INEGI estándar de 5 dígitos = entidad (2) + municipio (3).
# Ejemplo: Cuernavaca = 17007. Es la llave con la que este resultado podría
# cruzarse después con población, superficie o cualquier otra fuente del INEGI.
df_sel['cve_geo'] = (df_sel['cve_ent'].astype(str).str.zfill(2)
                     + df_sel['cve_mun'].astype(str).str.zfill(3))

print('Ejemplos de cve_geo:')
print(df_sel[['cve_geo', 'municipio']].drop_duplicates().sort_values('cve_geo').head(5).to_string(index=False))

In [ ]:
# Verificación de nulos en el dataset ya reducido
# Aquí sí esperamos cero, porque descartamos las columnas opcionales.
print('Nulos en columnas seleccionadas:')
print(df_sel.isnull().sum())


id               0
codigo_act       0
nombre_act       0
per_ocu          0
cve_ent          0
cve_mun          0
municipio        0
entidad          0
tipoUniEco       0
latitud          0
longitud         0
sector_codigo    0
sector_nombre    0
cve_geo          0
dtype: int64


In [ ]:
# El id identifica una unidad económica. Si estuviera repetido estaríamos
# contando dos veces el mismo negocio.
duplicados_id = df_sel.duplicated(subset=['id']).sum()
print('Duplicados por ID:', duplicados_id)

if duplicados_id > 0:
    df_sel = df_sel.drop_duplicates(subset=['id'])
    print('Duplicados por ID eliminados. Nuevo shape:', df_sel.shape)
else:
    print('No hay duplicados por ID.')

In [ ]:
# Reseteamos el índice para que quede continuo después de la limpieza
df_sel = df_sel.reset_index(drop=True)
print('Shape final del dataset limpio:', df_sel.shape)
df_sel.head()

---
## 7. Resumen de la limpieza realizada

In [ ]:
# Comparación explícita antes / después, para poder defender la limpieza
# con números y no con la frase "limpiamos los datos".
print('Antes de limpiar')
print('Filas x columnas:', df.shape)
print('Duplicados completos:', df.duplicated().sum())
print('Columnas con nulos:', (df.isnull().sum() > 0).sum())
print()
print('Después de limpiar')
print('Filas x columnas:', df_sel.shape)
print('Duplicados por id:', df_sel.duplicated(subset=['id']).sum())
print('Columnas con nulos:', (df_sel.isnull().sum() > 0).sum())
print('Municipios:', df_sel['municipio'].nunique())
print('Sectores SCIAN:', df_sel['sector_nombre'].nunique())

  Duplicados completos: 0
  Columnas con nulos  : 24

DESPUÉS DE LIMPIAR
  Filas x columnas    : (113066, 14)
  Duplicados por id   : 0
  Columnas con nulos  : 0
  Municipios          : 36
  Sectores SCIAN      : 20


**Resumen de la limpieza:**

1. **Dataset original:** 113,066 filas × 42 columnas del DENUE (INEGI) para Morelos.
2. **Duplicados:** se revisaron filas completas y por `id`, no se encontró ninguno.
3. **Nulos:** se identificaron columnas con altos porcentajes de nulos, todas ellas campos opcionales
   de domicilio y contacto. Las columnas que usamos no tienen nulos, así que no fue necesario imputar
   ningún valor.
4. **Selección de columnas:** de 42 a 11 columnas relevantes. Se descartaron domicilio, contacto,
   AGEB y manzana por describir la ubicación exacta y no la actividad económica.
5. **Variables creadas:** `sector_codigo` y `sector_nombre` (sector SCIAN de 2 dígitos) y `cve_geo`
   (clave INEGI de 5 dígitos).
6. **No se eliminó ninguna fila.** Los 113,066 registros llegan completos al análisis.

---
## 8. Exportar el dataset limpio

In [ ]:
# Guardamos en UTF-8 para que el archivo procesado sea legible.
RUTA_PROCESSED.mkdir(parents=True, exist_ok=True)
archivo_limpio = RUTA_PROCESSED / 'dataset_limpio.csv'

df_sel.to_csv(archivo_limpio, index=False, encoding='utf-8')
print('Dataset limpio guardado en:', archivo_limpio.relative_to(BASE))
print('Shape final:', df_sel.shape)

---
## 9. Construcción de la tabla municipal

### Decisión central del proyecto

El dataset es un directorio de establecimientos, pero la pregunta guía es sobre municipios.
Por eso el siguiente paso es agregar los 113,066 registros a una tabla de 36 filas, una por municipio.

### ¿Perfil con conteos o con porcentajes?

Usamos el porcentaje de unidades económicas de cada grupo de sector sobre el total del municipio,
no el conteo absoluto.

**¿Por qué?** Cuernavaca tiene 25,404 unidades y Coatlán del Río 235. Si usáramos conteos absolutos,
KMeans separaría por tamaño y los grupos resultantes serían "municipios grandes / medianos / chicos",
que es una respuesta sobre el tamaño de la economía, no sobre su perfil. Con porcentajes, un municipio
chico especializado en manufactura puede quedar junto a otro chico igual de especializado, y separado de
uno chico dedicado al comercio.

El total absoluto lo conservamos como variable descriptiva (para caracterizar los grupos después),
pero fuera del modelo.

In [ ]:
# Agrupamos los 20 sectores SCIAN en 8 grupos.
# Motivo: hay sectores con 2 registros (Corporativos) o 30 (Minería) en todo el
# estado. Como variables serían casi constantes, no aportan a separar municipios
# y sí agregan ruido. Los grupos se arman juntando sectores de naturaleza parecida.
GRUPOS_SECTOR = {
    # Comercio al por menor: es el sector dominante, va solo
    'Comercio al por menor':    'com_menor',
    # Comercio al por mayor: lógica distinta (abasto/intermediación), va solo
    'Comercio al por mayor':    'com_mayor',
    # Manufactura: los tres códigos SCIAN ya venían unificados
    'Manufactura':              'manufactura',
    # Alojamiento y alimentos: hoteles y restaurantes -> proxy de turismo
    'Alojamiento y alimentos':  'aloj_alim',
    # Otros servicios: reparación, estéticas, lavanderías -> servicios personales
    'Otros servicios':          'otros_serv',
    # Salud y educación juntos: ambos son servicios sociales de cobertura
    'Servicios de salud':       'salud_educ',
    'Servicios educativos':     'salud_educ',
    # Servicios especializados: requieren capital humano o financiero
    'Servicios profesionales':  'serv_espec',
    'Servicios financieros':    'serv_espec',
    'Servicios inmobiliarios':  'serv_espec',
    'Servicios de apoyo':       'serv_espec',
    'Información y medios':     'serv_espec',
    'Corporativos':             'serv_espec',
    # Grupo residual: sectores con muy pocos establecimientos en el DENUE
    'Agricultura y ganadería':  'otros_grupo',
    'Minería':                  'otros_grupo',
    'Electricidad y agua':      'otros_grupo',
    'Construcción':             'otros_grupo',
    'Transporte':               'otros_grupo',
    'Gobierno':                 'otros_grupo',
    'Esparcimiento y cultura':  'otros_grupo',
}

df_sel['grupo_sector'] = df_sel['sector_nombre'].map(GRUPOS_SECTOR)

# assert: si un sector quedara sin grupo, los porcentajes ya no sumarían 100
assert df_sel['grupo_sector'].isnull().sum() == 0, 'Hay sectores sin grupo asignado'

print('Unidades económicas por grupo de sector:')
print(df_sel['grupo_sector'].value_counts())

In [ ]:
# per_ocu es un rango, no un número. Para poder estimar el tamaño promedio de
# los establecimientos le asignamos a cada rango su punto medio aproximado.
# Es una estimación, no empleo medido: se declara en limitaciones.
PUNTO_MEDIO_PER_OCU = {
    '0 a 5 personas':       3,
    '6 a 10 personas':      8,
    '11 a 30 personas':    20,
    '31 a 50 personas':    40,
    '51 a 100 personas':   75,
    '101 a 250 personas': 175,
    '251 y más personas': 300,   # rango abierto: 300 es un supuesto conservador
}

df_sel['empleo_estimado'] = df_sel['per_ocu'].map(PUNTO_MEDIO_PER_OCU)
assert df_sel['empleo_estimado'].isnull().sum() == 0, 'Hay rangos de per_ocu sin punto medio'

# Marcamos los establecimientos semifijos (puestos, tianguis) como 0/1
# para poder promediarlos por municipio más adelante.
df_sel['es_semifijo'] = df_sel['tipoUniEco'].apply(lambda x: 1 if x == 'Semifijo' else 0)

print('Empleo estimado total en Morelos:', df_sel['empleo_estimado'].sum())
print('Promedio de personas por unidad  :', round(df_sel['empleo_estimado'].mean(), 2))

In [ ]:
# crosstab construye directamente la matriz municipio x grupo de sector:
# cada celda es el número de unidades económicas de ese grupo en ese municipio.
orden_grupos = ['com_menor', 'com_mayor', 'manufactura', 'aloj_alim',
                'otros_serv', 'salud_educ', 'serv_espec', 'otros_grupo']

matriz_conteos = pd.crosstab(df_sel['municipio'], df_sel['grupo_sector'])
matriz_conteos = matriz_conteos[orden_grupos]   # orden fijo para todas las salidas

print('Shape de la matriz municipio x sector:', matriz_conteos.shape)
matriz_conteos.head()

In [ ]:
# Guardamos la matriz de conteos como evidencia intermedia del análisis
RUTA_OUTPUTS.mkdir(parents=True, exist_ok=True)
matriz_conteos.to_csv(RUTA_OUTPUTS / 'matriz_municipio_sector.csv', encoding='utf-8')
print('Guardado: outputs/matriz_municipio_sector.csv')

In [ ]:
# Convertimos los conteos a porcentaje por municio (cada fila suma 100).
# div(..., axis=0) divide cada fila entre su propio total.
matriz_pct = matriz_conteos.div(matriz_conteos.sum(axis=1), axis=0) * 100
matriz_pct.columns = ['pct_' + c for c in matriz_pct.columns]

# assert: verificamos que efectivamente cada municipio sume 100%
assert (matriz_pct.sum(axis=1).round(6) == 100).all(), 'Alguna fila no suma 100%'

print('Verificación: todas las filas suman', matriz_pct.sum(axis=1).round(2).unique(), '%')
matriz_pct.round(2).head()

In [ ]:
# Armamos la tabla municipal final: composición porcentual + variables estructurales.
df_mun = matriz_pct.copy()

# Variables descriptivas, no entran al modelo
df_mun['total_unidades']  = matriz_conteos.sum(axis=1)
df_mun['empleo_estimado'] = df_sel.groupby('municipio')['empleo_estimado'].sum()

# Variables estructurales, sí entran al modelo
# tam_prom_unidad: ¿son negocios chicos o grandes? Distingue una economía de
# changarros de una con establecimientos formales de mayor tamaño.
df_mun['tam_prom_unidad'] = df_mun['empleo_estimado'] / df_mun['total_unidades']

# pct_semifijo: proxy grueso de comercio en vía pública / informalidad de local.
df_mun['pct_semifijo'] = df_sel.groupby('municipio')['es_semifijo'].mean() * 100

# Variables de apoyo para el mapa, no entran al modelo
# Centroide de los establecimientos del municipio: nos permite dibujar el mapa
# sin necesidad de librerías de cartografía ni archivos de geometría.
df_mun['lat_prom'] = df_sel.groupby('municipio')['latitud'].mean()
df_mun['lon_prom'] = df_sel.groupby('municipio')['longitud'].mean()

# Clave INEGI, para trazabilidad del resultado
df_mun['cve_geo'] = df_sel.groupby('municipio')['cve_geo'].first()

print('Shape de la tabla municipal:', df_mun.shape)
print('Nulos totales:', df_mun.isnull().sum().sum())
df_mun.head()

In [ ]:
# Estadísticas descriptivas de la tabla municipal.
# Esta tabla ya es el insumo del modelo: 36 observaciones.
df_mun.describe().round(2)

---
## 10. Análisis exploratorio

Dos gráficas, cada una con un propósito concreto:

1. **Gráfica 1** justifica una decisión metodológica (por qué no usamos conteos absolutos).
2. **Gráfica 2** muestra dónde está realmente la señal que el modelo va a aprovechar.

In [ ]:
# Etiquetas y colores compartidos por todas las gráficas del notebook.
# Los definimos una sola vez para que el color del cluster 2 sea el mismo
# en las gráficas 4, 5 y 6 y no haya que reinterpretar la leyenda cada vez.
COLORES_CLUSTER = ['#C44E52', '#4C72B0', '#55A868', '#8172B2']

# Los nombres internos (pct_com_menor) sirven para programar, pero no para
# comunicar. Este diccionario los traduce en las etiquetas de las gráficas.
NOMBRES_VARS = {
    'pct_com_menor':   'Comercio al por menor',
    'pct_com_mayor':   'Comercio al por mayor',
    'pct_manufactura': 'Manufactura',
    'pct_aloj_alim':   'Alojamiento y alimentos',
    'pct_otros_serv':  'Otros servicios',
    'pct_salud_educ':  'Salud y educación',
    'pct_serv_espec':  'Servicios especializados',
    'pct_otros_grupo': 'Otros sectores',
    'tam_prom_unidad': 'Tamaño prom. por unidad',
    'pct_semifijo':    'Establecimientos semifijos',
}
print('Etiquetas y colores definidos.')

### Gráfica 1 — Total de unidades económicas por municipio

**¿Qué vamos a ver?** El número absoluto de establecimientos en cada uno de los 36 municipios,
ordenados de mayor a menor. Esperamos una distribución muy desigual.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 10))

# Gráfica de barras horizontales: con 36 municipios las etiquetas
# solo son legibles en horizontal.
df_mun['total_unidades'].sort_values().plot(
    kind='barh', ax=ax, color='#4C72B0', edgecolor='white', linewidth=0.5
)

ax.set_title('Unidades económicas por municipio en Morelos (DENUE, INEGI)', fontsize=12, pad=12)
ax.set_xlabel('Número de unidades económicas')
ax.set_ylabel('Municipio')
ax.grid(axis='y', visible=False)

# Anotamos la concentración: es el argumento de la gráfica
top3 = df_mun['total_unidades'].sort_values(ascending=False).head(3)
concentracion = top3.sum() / df_mun['total_unidades'].sum() * 100
ax.text(0.97, 0.05,
        f'{", ".join(top3.index)}\nconcentran el {concentracion:.1f}%\ndel total estatal',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#FFF3CD', edgecolor='#B8860B', alpha=0.9))

plt.tight_layout()
RUTA_IMAGENES.mkdir(parents=True, exist_ok=True)
plt.savefig(RUTA_IMAGENES / 'grafica_1_unidades_por_municipio.png', bbox_inches='tight')
plt.show()

**¿Qué estamos viendo y por qué importa?**

La distribución es extremadamente desigual: **Cuernavaca, Cuautla y Jiutepec concentran el 45% de
todas las unidades económicas del estado**, mientras que Coatlán del Río tiene 235 y Tlalnepantla 263.
Cuernavaca sola tiene **108 veces** más establecimientos que Coatlán del Río.

**Por qué importa para el modelo:** esta gráfica es la justificación empírica de la decisión de la
sección 9. Si alimentáramos KMeans con estos conteos absolutos, la distancia euclidiana entre
Cuernavaca y cualquier otro municipio sería enorme, y el algoritmo agruparía por **tamaño**, no por
**perfil**. Los grupos resultantes serían triviales ("grandes / medianos / chicos") y no responderían
la pregunta guía. Por eso el modelo trabaja con **porcentajes**.

### Gráfica 2 — Composición sectorial de cada municipio

**¿Qué vamos a ver?** Cómo se reparte el 100% de las unidades económicas de cada municipio entre los
8 grupos de sector. Ordenamos por porcentaje de comercio al por menor para que se aprecie el rango.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

columnas_pct = ['pct_' + c for c in orden_grupos]

# Barras apiladas al 100%: es la forma directa de comparar COMPOSICIÓN
# entre unidades de tamaños muy distintos.
# rename(): la leyenda debe leerse sin conocer los nombres internos.
df_mun.sort_values('pct_com_menor')[columnas_pct].rename(columns=NOMBRES_VARS).plot(
    kind='barh', stacked=True, ax=ax, colormap='tab20', width=0.8
)

ax.set_title('Composición sectorial de las unidades económicas por municipio',
             fontsize=12, pad=12)
ax.set_xlabel('Porcentaje de las unidades económicas del municipio (%)')
ax.set_ylabel('Municipio')
ax.set_xlim(0, 100)
ax.grid(axis='y', visible=False)
ax.legend(title='Grupo de sector', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)

plt.tight_layout()
plt.savefig(RUTA_IMAGENES / 'grafica_2_composicion_sectorial.png', bbox_inches='tight')
plt.show()

**¿Qué estamos viendo y por qué importa?**

Tres lecturas:

1. **El comercio al por menor domina en todos los municipios sin excepción** (entre 28.8% y 56.2%).
   No es una variable que distinga por sí sola: todos los municipios tienen mucho comercio.
2. **La diferencia está en el margen.** Los casos visibles son Temoac, con una franja de manufactura
   desproporcionada (45.3% contra 10.5% de media estatal), y Tepoztlán, con la franja de alojamiento
   y alimentos más ancha del estado (26.4% contra 14.7%).
3. **La señal existe pero es sutil.** Salvo Temoac, las barras se parecen bastante entre sí. Esto
   anticipa que los grupos que encuentre KMeans van a estar separados por diferencias de pocos
   puntos porcentuales, no por contrastes dramáticos — y por lo tanto que la silhouette será baja.

Es exactamente el tipo de estructura fina que un modelo de agrupamiento sirve para detectar y que
sería muy difícil ver a simple vista en 36 municipios × 8 sectores.

In [ ]:
# Tabla complementaria: los extremos de las variables más informativas.
# Sirve para poner nombre y apellido a lo que se ve en la gráfica 2.
for variable in ['pct_manufactura', 'pct_aloj_alim', 'pct_com_menor']:
    print('=' * 60)
    print(variable, ' | media estatal:', round(df_mun[variable].mean(), 2), '%')
    print('  TOP 5:')
    print(df_mun[variable].sort_values(ascending=False).head(5).round(2).to_string())
    print('  BOTTOM 5:')
    print(df_mun[variable].sort_values().head(5).round(2).to_string())
    print()

---
## 11. Selección de variables y escalado

### Las 9 variables del modelo

**Composición sectorial (7):**
`pct_com_menor`, `pct_com_mayor`, `pct_manufactura`, `pct_aloj_alim`, `pct_otros_serv`,
`pct_salud_educ`, `pct_serv_espec`

**Estructura de los establecimientos (2):**
`tam_prom_unidad`, `pct_semifijo`

### Correspondencia con la pregunta de investigación

La pregunta habla de tres familias de actividad: **manufactura, comercio y servicios**. Las 7 variables
de composición las cubren por completo, con el nivel de detalle que el DENUE permite:

| Familia de la pregunta | Variables del modelo | % estatal |
|---|---|---|
| **Manufactura** | `pct_manufactura` | 8.8 |
| **Comercio** | `pct_com_menor` + `pct_com_mayor` | 46.9 |
| **Servicios** | `pct_aloj_alim` + `pct_otros_serv` + `pct_salud_educ` + `pct_serv_espec` | 40.7 |

No usamos una sola variable por familia porque eso borraría lo que más distingue a los municipios:
"servicios" agrupa desde una fonda hasta un despacho contable, y esa diferencia es justo la señal.
Por eso el comercio se separa en menor/mayor y los servicios en cuatro tipos.

### Justificación de las tres decisiones que hay que poder defender

**1. ¿Por qué se excluye `pct_otros_grupo`?**
Porque los porcentajes de un municipio **suman 100**: es un dato *composicional*. Si conocemos siete
de los ocho grupos, el octavo queda determinado — es **información redundante**. Incluirlo le daría
peso doble a esa dimensión. Dejamos fuera el grupo residual porque es el menos interpretable
(mezcla agricultura, minería, gobierno y transporte). Aun así, las variables restantes **siguen sin ser
independientes entre sí**, y eso se declara en limitaciones.

**2. ¿Por qué no entra `total_unidades`?**
Es la variable que mide **tamaño**, no perfil (ver gráfica 1). Se conserva solo para **describir** los
grupos una vez formados.

**3. ¿Por qué sí entran `tam_prom_unidad` y `pct_semifijo`?**
Porque dos municipios pueden tener la misma composición sectorial y economías muy distintas: uno de
establecimientos grandes y formales, otro de changarros y puestos semifijos. Estas dos variables
capturan esa diferencia estructural que la composición sectorial por sí sola no ve.

In [ ]:
# Lista explícita de las variables del modelo. Todo lo demás en df_mun
# es descriptivo y NO se le da al algoritmo.
VARS = [
    'pct_com_menor',    # comercio al por menor
    'pct_com_mayor',    # comercio al por mayor / abasto
    'pct_manufactura',  # manufactura
    'pct_aloj_alim',    # alojamiento y alimentos (proxy de turismo)
    'pct_otros_serv',   # servicios personales
    'pct_salud_educ',   # salud y educación
    'pct_serv_espec',   # servicios especializados
    'tam_prom_unidad',  # tamaño promedio del establecimiento
    'pct_semifijo',     # % de establecimientos semifijos
]

X = df_mun[VARS]
print('Matriz del modelo:', X.shape, '-> 36 municipios x 9 variables')

In [ ]:
# Evidencia de por qué el escalado es OBLIGATORIO:
# comparar la media y la desviación de las 9 variables.
print('ANTES DE ESCALAR')
print(X.describe().loc[['mean', 'std', 'min', 'max']].round(2).to_string())

**Por qué hay que escalar.** En la tabla de arriba, `pct_com_menor` ronda 45 y `tam_prom_unidad` ronda 4.4.
KMeans agrupa usando **distancia euclidiana**: sin escalar, una diferencia de 5 puntos en comercio al por
menor pesaría lo mismo que una diferencia de 5 personas por establecimiento, que en esa variable es
enorme. En la práctica, el comercio al por menor **dominaría** la distancia y las demás variables serían
casi irrelevantes.

`StandardScaler` deja todas las variables con **media 0 y desviación 1**, de modo que cada una aporta
lo mismo a la distancia.

In [ ]:
# StandardScaler: (valor - media) / desviación estándar, columna por columna
escalador = StandardScaler()
X_scaled = escalador.fit_transform(X)

# Verificamos el resultado del escalado
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=VARS)
print('DESPUÉS DE ESCALAR')
print(X_scaled_df.describe().loc[['mean', 'std', 'min', 'max']].round(2).to_string())

In [ ]:
# Matriz de correlación entre las 9 variables del modelo.
# La revisamos para anticipar la pregunta sobre redundancia entre variables.
correlacion = X.corr().round(2)
correlacion

**Lectura de la matriz de correlación.** Las correlaciones más fuertes son:

- `pct_manufactura` ↔ `pct_com_menor`: **−0.51**. Es el efecto composicional esperado — si un municipio
  dedica mucho a manufactura, necesariamente le queda menos porcentaje para comercio.
- `tam_prom_unidad` ↔ `pct_otros_serv`: **+0.55**, y ↔ `pct_salud_educ`: **+0.45**. Los municipios con
  más servicios tienden a tener establecimientos más grandes.

Ninguna pasa de |0.55|, así que **no hay dos variables que digan literalmente lo mismo** y no eliminamos
ninguna. Pero las correlaciones no son cero, y esa dependencia parcial es inherente a los datos
composicionales: **se declara como limitación**.

In [ ]:
# Guardamos las 9 variables del modelo como evidencia reproducible
X.round(4).to_csv(RUTA_OUTPUTS / 'variables_modelo.csv', encoding='utf-8')
print('Guardado: outputs/variables_modelo.csv', X.shape)

---
## 12. Elección del número de grupos (k)

KMeans **exige** que le digamos cuántos grupos buscar. Evaluamos k = 2 … 8 con dos criterios:

- **Inercia (método del codo):** suma de distancias al cuadrado de cada municipio a su centroide.
  Siempre baja al aumentar k, así que buscamos el punto donde **deja de bajar rápido**.
- **Silhouette:** de −1 a 1. Mide qué tan bien separado está cada municipio de los grupos vecinos.
  Más alto es mejor.

### Regla de decisión (declarada ANTES de ver los resultados)

Tomamos el k con **mejor silhouette**. Si los dos criterios apuntan a k distintos, rompemos el empate
con el **codo** y con la **interpretabilidad** de los grupos, y lo decimos explícitamente — nunca
justificando el k después de haber visto qué grupos salieron.

`random_state=42` y `n_init=10` fijos en todas las corridas, para que el resultado sea reproducible.

In [ ]:
# Probamos k = 2 a 8 y guardamos las dos métricas de cada corrida.
# n_init=10: KMeans se inicializa 10 veces y se queda con la mejor solución,
# lo que reduce (no elimina) la sensibilidad a la inicialización aleatoria.
resultados_k = []

for k in range(2, 9):
    modelo_k = KMeans(n_clusters=k, random_state=42, n_init=10)
    etiquetas_k = modelo_k.fit_predict(X_scaled)
    resultados_k.append({
        'k': k,
        'inercia': round(modelo_k.inertia_, 2),
        'silhouette': round(silhouette_score(X_scaled, etiquetas_k), 4),
        'tamanos': str(sorted(pd.Series(etiquetas_k).value_counts().tolist(), reverse=True)),
    })

metricas_k = pd.DataFrame(resultados_k)

# Δ inercia: cuánto baja la inercia al pasar de k-1 a k. El "codo" es donde
# esta caída se vuelve pequeña.
metricas_k['delta_inercia'] = metricas_k['inercia'].diff().round(2)

metricas_k

In [ ]:
# Guardamos las métricas como evidencia de que el k se eligió con números
metricas_k.to_csv(RUTA_OUTPUTS / 'metricas_k.csv', index=False, encoding='utf-8')
print('Guardado: outputs/metricas_k.csv')

In [ ]:
# k elegido. Se define aquí, en una sola variable, para que todo lo que sigue
# dependa de ella y no haya números "4" escritos a mano por el notebook.
K_ELEGIDO = 4
k_mejor_sil = int(metricas_k.loc[metricas_k['silhouette'].idxmax(), 'k'])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Panel 1: método del codo
metricas_k.plot(x='k', y='inercia', marker='o', ax=axes[0], legend=False, color='#4C72B0')
axes[0].set_title('1. Método del codo')
axes[0].set_xlabel('Número de grupos (k)')
axes[0].set_ylabel('Inercia (suma de distancias al cuadrado)')
axes[0].axvline(K_ELEGIDO, color='#C44E52', linestyle='--', linewidth=1.5)

# Panel 2: caída de la inercia
# En la curva de arriba el codo casi no se aprecia porque la caída es gradual.
# Graficando la DIFERENCIA entre k y k-1 se ve de forma directa en qué punto
# agregar un grupo deja de aportar. Así el argumento se ve, no solo se afirma.
caidas = metricas_k.dropna(subset=['delta_inercia'])
barras = axes[1].bar(caidas['k'], -caidas['delta_inercia'],
                     color=['#C44E52' if k == K_ELEGIDO else '#B0B0B0' for k in caidas['k']],
                     edgecolor='white')
axes[1].set_title('2. Cuánto baja la inercia al agregar un grupo')
axes[1].set_xlabel('Número de grupos (k)')
axes[1].set_ylabel('Caída de la inercia respecto a k-1')
axes[1].bar_label(barras, fmt='%.1f', fontsize=8, padding=2)
axes[1].grid(axis='x', visible=False)

# --- Panel 3: silhouette ---
metricas_k.plot(x='k', y='silhouette', marker='o', ax=axes[2], legend=False, color='#55A868')
axes[2].set_title('3. Coeficiente de silhouette')
axes[2].set_xlabel('Número de grupos (k)')
axes[2].set_ylabel('Silhouette (más alto = mejor separación)')
axes[2].axvline(K_ELEGIDO, color='#C44E52', linestyle='--', linewidth=1.5)

# Marcamos también el máximo de silhouette, que NO coincide con el k elegido:
# ocultarlo sería deshonesto, así que queda visible en la gráfica.
axes[2].axvline(k_mejor_sil, color='#8172B2', linestyle=':', linewidth=1.5)
axes[2].legend(['silhouette', f'k elegido = {K_ELEGIDO}',
                f'máx. silhouette = {k_mejor_sil}'], fontsize=8)

fig.suptitle('Selección del número de grupos: los dos criterios discrepan', fontsize=13)
plt.tight_layout()
plt.savefig(RUTA_IMAGENES / 'grafica_3_seleccion_k.png', bbox_inches='tight')
plt.show()

In [ ]:
# Dejamos por escrito, con números, la discrepancia entre los dos criterios
print('Máximo de silhouette en k =', k_mejor_sil,
      '->', metricas_k.loc[metricas_k['k'] == k_mejor_sil, 'silhouette'].values[0])
print('Mayor caída de inercia al pasar a k =',
      int(metricas_k.loc[metricas_k['delta_inercia'].idxmin(), 'k']),
      '->', metricas_k['delta_inercia'].min())
print()
print('Tamaños de los grupos en cada k:')
print(metricas_k[['k', 'silhouette', 'tamanos']].to_string(index=False))

### Decisión: k = 4

**Los dos criterios discrepan, y hay que decirlo.**

| Criterio | Apunta a | Valor |
|---|---|---|
| Silhouette máxima | **k = 3** | 0.2989 |
| Mayor caída de inercia (codo) | **k = 4** | −44.88 (la mayor de toda la serie) |

Aplicamos la regla declarada arriba y rompimos el empate con el codo y la interpretabilidad:

1. **El codo apunta a k = 4.** En el panel 1 la curva de inercia baja de forma tan gradual que el codo
   casi no se aprecia; por eso el **panel 2** grafica directamente cuánto baja la inercia en cada paso.
   Ahí se ve que la caída al pasar de 3 a 4 grupos (−44.88) es la **mayor de toda la serie**, y que
   **ningún k posterior vuelve a alcanzarla** (la mayor después de k = 4 es −32.94, en k = 6). Es decir:
   el cuarto grupo todavía explica estructura real.
2. **k = 3 produce grupos desbalanceados hasta el punto de ser poco informativos:** deja **31 de los
   36 municipios en un solo grupo**. Un resultado que dice "31 municipios se parecen entre sí" no
   responde la pregunta guía. Con k = 4 los tamaños son 17 / 11 / 7 / 1.
3. **Temoac queda solo en k = 3, 4, 5, 6, 7 y 8.** No es un artefacto de haber elegido k = 4: es un
   municipio genuinamente distinto al resto (ver sección 15).

**Lo decimos con todas sus letras:** la silhouette de k = 4 es **0.2074**, un valor **bajo**. Significa
que los grupos existen pero **no están nítidamente separados**, algo consistente con lo que ya vimos en
la gráfica 2. Esta es una limitación real del análisis y se retoma en la sección 16.

**También probamos la alternativa obvia** —tratar a Temoac como valor atípico y excluirlo— y **no
mejora**: la silhouette baja a ~0.21 y Tlalnepantla se convierte en el nuevo grupo de un solo municipio.
Por eso conservamos los 36 municipios.

---
## 13. Aplicación del modelo: KMeans con k = 4

In [ ]:
# Ajustamos el modelo definitivo sobre las 9 variables ESCALADAS.
modelo_kmeans = KMeans(n_clusters=K_ELEGIDO, random_state=42, n_init=10)
etiquetas = modelo_kmeans.fit_predict(X_scaled)

print('Modelo ajustado.')
print('Inercia final    :', round(modelo_kmeans.inertia_, 2))
print('Silhouette final :', round(silhouette_score(X_scaled, etiquetas), 4))
print('Iteraciones      :', modelo_kmeans.n_iter_)

In [ ]:
# KMeans numera los grupos de forma arbitraria: el "cluster 0" de una corrida
# puede ser el "cluster 2" de otra. Los reetiquetamos ordenándolos por su
# porcentaje medio de comercio al por menor, para que la numeración sea
# ESTABLE y podamos hablar de "el cluster 0" sin ambigüedad.
df_mun['cluster'] = etiquetas

orden_clusters = df_mun.groupby('cluster')['pct_com_menor'].mean().sort_values().index
mapa_clusters = {viejo: nuevo for nuevo, viejo in enumerate(orden_clusters)}
df_mun['cluster'] = df_mun['cluster'].map(mapa_clusters)

print('Reetiquetado aplicado (orden ascendente de pct_com_menor):')
print(mapa_clusters)

In [ ]:
# Tamaño de cada grupo
print('Municipios por grupo:')
print(df_mun['cluster'].value_counts().sort_index().to_string())
print()

# Lista completa: es el resultado que se presenta en la exposición
for c in sorted(df_mun['cluster'].unique()):
    municipios_c = sorted(df_mun[df_mun['cluster'] == c].index)
    print(f'--- CLUSTER {c}  ({len(municipios_c)} municipios) ---')
    print('  ' + ', '.join(municipios_c))
    print()

In [ ]:
# Exportamos el resultado a nivel municipio (requerido por la rúbrica).
resultados_modelo = df_mun.reset_index()[
    ['cve_geo', 'municipio', 'cluster'] + VARS + ['total_unidades']
].sort_values(['cluster', 'municipio'])

resultados_modelo.round(4).to_csv(
    RUTA_OUTPUTS / 'resultados_modelo.csv', index=False, encoding='utf-8'
)
print('Guardado: outputs/resultados_modelo.csv', resultados_modelo.shape)
resultados_modelo.head()

---
## 14. Caracterización de los grupos

Un grupo no significa nada hasta que sabemos **qué lo distingue**. Aquí comparamos la media de cada
variable dentro de cada grupo contra la **media estatal**.

In [ ]:
# Medias por grupo: es la tabla de resultados del modelo.
resumen_clusters = df_mun.groupby('cluster')[VARS + ['total_unidades']].mean().round(2)
resumen_clusters.insert(0, 'n_municipios', df_mun['cluster'].value_counts().sort_index())

# Agregamos la lista de municipios de cada grupo en una sola celda
resumen_clusters['municipios'] = (
    df_mun.reset_index().groupby('cluster')['municipio']
    .apply(lambda s: ', '.join(sorted(s)))
)

resumen_clusters.to_csv(RUTA_OUTPUTS / 'resumen_resultados.csv', encoding='utf-8')
print('Guardado: outputs/resumen_resultados.csv')
resumen_clusters[['n_municipios'] + VARS + ['total_unidades']]

In [ ]:
# Desviación de cada grupo respecto a la MEDIA ESTATAL.
# Esta es la tabla que realmente permite interpretar: dice en qué se
# diferencia cada grupo, no solo qué valores tiene.
media_estatal = df_mun[VARS].mean()
desviaciones = (df_mun.groupby('cluster')[VARS].mean() - media_estatal).round(2)

print('Media estatal de referencia:')
print(media_estatal.round(2).to_string())
print()
print('Diferencia de cada grupo respecto a la media estatal:')
desviaciones

### Gráfica 4 — Perfil de cada grupo

**¿Qué vamos a ver?** La diferencia de cada grupo respecto a la media estatal, variable por variable.
Las barras hacia arriba indican "este grupo tiene más de lo normal"; hacia abajo, "menos de lo normal".
Es **la gráfica principal de resultados**.

La dibujamos en **dos paneles** por una razón práctica: la desviación de Temoac en manufactura es tan
grande que, en una sola escala, deja a los otros tres grupos aplastados contra el eje y no se puede leer
nada de ellos. El panel B repite exactamente la misma información sin el cluster 0.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# Graficamos las DESVIACIONES y no las medias: así se ve de un vistazo
# qué distingue a cada grupo, que es justo lo que hay que interpretar.
tabla_grafica = desviaciones.rename(columns=NOMBRES_VARS).T

# --- Panel A: los cuatro grupos ---
tabla_grafica.plot(kind='bar', ax=axes[0], width=0.8, edgecolor='white',
                   color=COLORES_CLUSTER)
axes[0].set_title('A. Los cuatro grupos (escala completa)', fontsize=11)

# Panel B: sin el cluster 0
# La barra de manufactura de Temoac (+34.8) es tan grande que aplasta a los
# otros tres grupos y los vuelve ilegibles. Repetimos la gráfica sin él para
# poder VER las diferencias sutiles, que son el resultado interesante.
tabla_grafica.drop(columns=0).plot(kind='bar', ax=axes[1], width=0.8,
                                   edgecolor='white', color=COLORES_CLUSTER[1:])
axes[1].set_title('B. Sin el cluster 0: la escala real de las diferencias', fontsize=11)

for ax in axes:
    ax.axhline(0, color='black', linewidth=1)
    ax.set_xlabel('Variable del modelo')
    ax.set_ylabel('Diferencia respecto a la media estatal')
    ax.legend(title='Cluster', fontsize=9)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='x', visible=False)
    for etiqueta in ax.get_xticklabels():
        etiqueta.set_ha('right')

fig.suptitle('Qué distingue a cada grupo: diferencia respecto a la media estatal',
             fontsize=13)
plt.tight_layout()
plt.savefig(RUTA_IMAGENES / 'grafica_4_perfil_clusters.png', bbox_inches='tight')
plt.show()

**¿Qué estamos viendo y por qué importa?**

**Panel A** deja clara la magnitud del caso Temoac: **+34.8 puntos** de manufactura sobre la media
estatal. Es un contraste de un orden de magnitud distinto al de todo lo demás — y el hecho de que el
panel A sea ilegible para los otros grupos *es en sí mismo el hallazgo*.

**Panel B** muestra la escala real en la que se distinguen los otros tres grupos: diferencias de
**1 a 3 puntos porcentuales**. El cluster 1 tiene más servicios personales y de salud/educación; el
cluster 2 destaca en comercio al por mayor; el cluster 3 en comercio al por menor y alojamiento y
alimentos, con menos servicios especializados.

**Por qué importa:** esta gráfica es la evidencia de que la agrupación **no es arbitraria** —cada grupo
tiene un patrón propio— pero también de que **las diferencias son sutiles**, lo que explica la
silhouette baja de la sección 12. Ambas cosas hay que decirlas.

### Gráfica 5 — Los grupos vistos en dos dimensiones (PCA)

**¿Qué vamos a ver?** Las 9 variables no se pueden dibujar en una hoja. PCA las comprime a 2 ejes que
conservan la mayor variabilidad posible, y ahí coloreamos por grupo.

**Importante — el orden de las operaciones:** el modelo se ajustó sobre las **9 variables escaladas**,
no sobre los componentes. PCA se usa **solo para poder ver el resultado**, después de agrupar.

In [ ]:
# PCA a 2 componentes, ÚNICAMENTE para visualizar.
pca = PCA(n_components=2, random_state=42)
componentes = pca.fit_transform(X_scaled)

varianza = pca.explained_variance_ratio_ * 100
print('Varianza explicada por PC1:', round(varianza[0], 1), '%')
print('Varianza explicada por PC2:', round(varianza[1], 1), '%')
print('Varianza acumulada        :', round(varianza.sum(), 1), '%')

df_mun['pc1'] = componentes[:, 0]
df_mun['pc2'] = componentes[:, 1]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# Dibujamos un scatter por cluster para que cada uno tenga su color y etiqueta
for c in sorted(df_mun['cluster'].unique()):
    sub = df_mun[df_mun['cluster'] == c]
    sub.plot(kind='scatter', x='pc1', y='pc2', ax=ax,
             color=COLORES_CLUSTER[c], s=70, alpha=0.85,
             label=f'Cluster {c} (n={len(sub)})', edgecolor='white', linewidth=0.8)

# Anotamos el nombre de cada municipio: sin esto la gráfica no se puede leer
for municipio, fila in df_mun.iterrows():
    ax.annotate(municipio, (fila['pc1'], fila['pc2']),
                fontsize=6.5, alpha=0.75, xytext=(4, 3), textcoords='offset points')

ax.set_title('Municipios de Morelos agrupados por perfil económico (proyección PCA)',
             fontsize=12, pad=12)
ax.set_xlabel(f'Componente principal 1 ({varianza[0]:.1f}% de la varianza)')
ax.set_ylabel(f'Componente principal 2 ({varianza[1]:.1f}% de la varianza)')
ax.legend(title='Grupo')

plt.tight_layout()
plt.savefig(RUTA_IMAGENES / 'grafica_5_pca_clusters.png', bbox_inches='tight')
plt.show()

**¿Qué estamos viendo y por qué importa?**

**Temoac aparece completamente aislado** en un extremo del gráfico. Los otros tres grupos ocupan
regiones distintas pero **con fronteras que se tocan**: hay municipios en el borde que podrían haber
caído en cualquiera de dos grupos.

**Advertencia necesaria:** los dos componentes solo conservan el **47.0%** de la varianza total. Es
decir, **más de la mitad de la información no se ve en esta gráfica**. Dos municipios que aquí se ven
cerca pueden estar lejos en las 9 dimensiones reales. Por eso esta gráfica sirve para *comunicar* el
resultado, no para *validarlo* — la validación son las métricas de la sección 12 y la tabla de
desviaciones de arriba.

### Gráfica 6 — Distribución territorial de los grupos

**¿Qué vamos a ver?** Los municipios ubicados por el centroide geográfico de sus establecimientos
(promedio de latitud y longitud), coloreados por grupo. Es un mapa aproximado que no requiere
librerías de cartografía.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))

for c in sorted(df_mun['cluster'].unique()):
    sub = df_mun[df_mun['cluster'] == c]
    sub.plot(kind='scatter', x='lon_prom', y='lat_prom', ax=ax,
             color=COLORES_CLUSTER[c], s=90, alpha=0.85,
             label=f'Cluster {c} (n={len(sub)})', edgecolor='white', linewidth=0.8)

for municipio, fila in df_mun.iterrows():
    ax.annotate(municipio, (fila['lon_prom'], fila['lat_prom']),
                fontsize=6.5, alpha=0.75, xytext=(4, 3), textcoords='offset points')

ax.set_title('Distribución territorial de los grupos en Morelos', fontsize=12, pad=12)
ax.set_xlabel('Longitud (centroide de los establecimientos)')
ax.set_ylabel('Latitud (centroide de los establecimientos)')
ax.legend(title='Grupo')

plt.tight_layout()
plt.savefig(RUTA_IMAGENES / 'grafica_6_mapa_clusters.png', bbox_inches='tight')
plt.show()

**¿Qué estamos viendo y por qué importa?**

Esta gráfica pone a prueba la idea de **"vecindarios económicos"** planteada en la justificación
(sección 0.1): ¿los municipios que se parecen económicamente son también vecinos en el mapa?

**La respuesta es: parcialmente.** Se alcanza a distinguir cierta lógica territorial —los municipios del
**cluster 1** se alinean sobre el corredor **Cuernavaca–Jiutepec–Yautepec–Cuautla** y las cabeceras del
sur, y buena parte del **cluster 2** se concentra en el **oriente** del estado, en la zona de los Altos—
pero **los grupos no son territorialmente compactos**. El cluster 1 incluye a Coatlán del Río y Tetecala,
en el extremo poniente, junto a Cuautla, en el oriente. Y el cluster 3 junta a Huitzilac (norte) con
Tepalcingo (sur), separados por todo el estado.

Es decir: **la similitud económica no se reduce a la cercanía geográfica**, que es justamente lo que
buscábamos comprobar.

**Advertencia importante:** esto es una **descripción**, no una explicación. La geografía **no entró al
modelo** — los grupos se formaron únicamente con composición sectorial y estructura de los
establecimientos. Que aparezca un patrón espacial parcial sugiere que perfil económico y ubicación están
relacionados, pero **no permite afirmar que uno cause el otro**.

In [ ]:
# "Municipio típico": el más cercano al centroide de su grupo en el espacio
# escalado de 9 dimensiones. Es el mejor representante de cada grupo y sirve
# para explicar cada cluster en pocos segundos durante la exposición.
print('MUNICIPIO MÁS REPRESENTATIVO DE CADA GRUPO')
print('(el más cercano al centro de su propio grupo)')
print()

for c in sorted(df_mun['cluster'].unique()):
    miembros = X_scaled_df[df_mun['cluster'] == c]
    centro = miembros.mean()
    distancias = ((miembros - centro) ** 2).sum(axis=1) ** 0.5
    print(f'  Cluster {c} (n={len(miembros)}): {distancias.idxmin()}')

---
## 15. Interpretación de los resultados

Aquí pasamos del resultado ("el municipio pertenece al cluster 2") a la interpretación
("qué caracteriza al cluster 2 y qué significa dentro del problema").

---

### Cluster 0 — El caso especializado: manufactura de confitería
**1 municipio:** Temoac. **Representante:** Temoac.

Se diferencia del resto de forma abrumadora por una sola variable: **45.3% de sus unidades económicas
son manufactura**, contra una media estatal de 10.5% (**+34.8 puntos**). A cambio tiene menos comercio
al por menor (−16.1) y menos servicios personales (−6.4) de lo normal.

Al revisar el detalle, **592 de sus 1,551 establecimientos** —el 38% del municipio— corresponden a una
sola actividad: *"Elaboración de dulces, chicles y productos de confitería que no sean de chocolate"*.
Esto es consistente con la conocida especialización de la región de Temoac en la producción de dulces
tradicionales.

Es el único grupo donde la especialización es tan marcada que el municipio **no se parece a ningún
otro del estado**. Que KMeans lo aísle no es un defecto del modelo: es el modelo detectando algo real.

---

### Cluster 1 — Perfil de servicios y establecimientos más grandes
**17 municipios:** Coatlán del Río, Cuautla, Cuernavaca, Emiliano Zapata, Jiutepec, Jojutla,
Jonacatepec de Leandro Valle, Mazatepec, Miacatlán, Puente de Ixtla, Temixco, Tetecala,
Tlaltizapán de Zapata, Tlaquiltenango, Xochitepec, Yautepec y Zacatepec.
**Representante:** Jojutla.

Se diferencia principalmente por tener **más servicios de lo normal**: servicios personales (+2.0),
salud y educación (+0.9) y servicios especializados (+0.4); y **menos manufactura** (−2.8). Además sus
establecimientos son en promedio **más grandes** (4.89 personas estimadas por unidad contra 4.43 estatal).

Es el grupo con diferencia más poblado de establecimientos: **5,084 unidades en promedio** por municipio,
contra 1,300–1,500 de los otros grupos. Concentra las tres ciudades principales del estado y las
cabeceras regionales del sur.

Los resultados sugieren que se trata de las **economías urbanas y de cabecera**, donde además del
comercio de siempre existe una oferta de servicios que requiere una población concentrada para
sostenerse.

---

### Cluster 2 — Comercio de abasto y manufactura ligera del oriente
**11 municipios:** Amacuzac, Atlatlahucan, Axochiapan, Ayala, Jantetelco, Ocuituco,
Tetela del Volcán, Tlalnepantla, Totolapan, Yecapixtla y Zacualpan de Amilpas.
**Representante:** Tetela del Volcán.

Se diferencia sobre todo por el **comercio al por mayor**: 3.85% contra 2.68% estatal (**+1.2 puntos**,
la desviación más alta de los cuatro grupos en esa variable). Tiene además algo más de manufactura
(+0.5) y **menos servicios personales** (−0.7), menos alojamiento y alimentos (−1.3) y establecimientos
más pequeños que el promedio (4.09 contra 4.43).

Con los datos disponibles se observa que es un perfil de **economía local de abasto**: municipios
predominantemente rurales del oriente del estado, con una función de intermediación comercial hacia su
entorno y sin la oferta de servicios de las cabeceras.

---

### Cluster 3 — Consumo final: comercio al por menor y alimentos
**7 municipios:** Coatetelco, Hueyapan, Huitzilac, Tepalcingo, Tepoztlán, Tlayacapan y Xoxocotla.
**Representante:** Tlayacapan.

Se diferencia por concentrar sus unidades económicas en **consumo directo**: comercio al por menor
(+3.1) y alojamiento y alimentos (+3.0), las dos desviaciones positivas más grandes fuera de Temoac.
Juntas suman **65.7%** de sus unidades, contra 59.6% de media estatal. En contrapartida, es el grupo con
**menos servicios especializados** (−1.3) y **menos salud y educación** (−1.7), y con la mayor proporción
de establecimientos **semifijos** (+0.6).

**Una observación honesta sobre este grupo:** mezcla municipios de vocación turística reconocida
(Tepoztlán, Tlayacapan, Huitzilac) con municipios rurales de fuerte identidad comunitaria (Hueyapan,
Coatetelco, Xoxocotla). El modelo los agrupa porque **su composición de unidades económicas se parece**,
no porque tengan la misma causa: en Tepoztlán el peso de alimentos y hospedaje viene del visitante, y en
Hueyapan el peso del comercio al por menor viene del abasto cotidiano local. **KMeans agrupa por
composición, no por causa**, y esta es una de las cosas que el análisis no puede distinguir.

---
## 16. Limitaciones

Reconocer hasta dónde llega el análisis es parte del análisis.

### Sobre los datos

1. **El DENUE cuenta establecimientos, no economía.** Registra que un negocio existe y a qué se dedica,
   pero **no su facturación, márgenes de ingreso ni valor agregado**. Un taller familiar y una planta
   industrial cuentan **como uno cada uno**. Un municipio con pocas unidades grandes puede tener más
   actividad económica real que uno con muchas unidades pequeñas, y este análisis no lo vería.
   Es la limitación declarada desde el planteamiento (sección 0.1, punto 4): las conclusiones se acotan
   a la **composición estructural del mercado formal**.

2. **`per_ocu` es un rango, no un conteo.** `tam_prom_unidad` se construyó asignándole a cada rango un
   **punto medio elegido por nosotros** (0–5 → 3, …, 251+ → 300). Es una **aproximación**, no empleo
   medido. El último rango es abierto, así que el 300 es un supuesto: si las unidades muy grandes
   fueran mucho mayores, subestimaríamos el tamaño de los municipios que las tienen.

3. **Corte temporal único (DENUE 2026/06).** Es una **fotografía**, no una película. No podemos hacer
   ninguna afirmación sobre evolución, crecimiento o tendencias: no sabemos si el perfil de un municipio
   es estable o si cambió recientemente.

4. **Sesgo de captura.** El DENUE cubre establecimientos **registrados con domicilio identificable**. La
   actividad informal ambulante y el trabajo desde el hogar quedan subrepresentados, y probablemente no
   de manera uniforme entre municipios: es plausible que el subregistro sea mayor en los municipios más
   rurales, lo que afectaría directamente su perfil.

5. **Variables ausentes.** No incorporamos población, superficie, PIB municipal ni ingreso. Por eso
   **no podemos hablar de densidad económica ni de bienestar**: solo de composición.

### Sobre el método

6. **Muy pocas observaciones para tantas variables.** 36 municipios y 9 variables es una razón alta.
   Con una n tan pequeña los grupos pueden ser **inestables**: quitar o agregar un municipio, o cambiar
   una variable, puede reacomodar la solución. Lo mitigamos con `n_init=10` y `random_state=42`
   (verificamos que dos corridas dan resultados idénticos), pero **no lo eliminamos**.

7. **Los datos son composicionales.** Los porcentajes de un municipio suman 100, así que las variables
   **no son independientes entre sí**: si una sube, otra necesariamente baja. Excluimos `pct_otros_grupo`
   para reducir la redundancia, pero la dependencia parcial permanece (ver la matriz de correlación de
   la sección 11) y KMeans no está diseñado para ese tipo de dato.

8. **La separación entre grupos es débil.** La silhouette del modelo final es **0.2074**, un valor bajo.
   Los grupos existen y tienen perfiles distinguibles, pero **no están nítidamente separados**: hay
   municipios en la frontera que podrían haber caído en otro grupo.

9. **Los criterios de selección de k discreparon.** La silhouette apuntaba a k = 3 y el codo a k = 4.
   Elegimos k = 4 y lo justificamos (sección 12), pero es una **decisión nuestra**, y con k = 3 la
   composición de los grupos sería distinta.

10. **KMeans siempre devuelve k grupos, existan o no.** El algoritmo asume grupos aproximadamente
    esféricos y de tamaño similar, supuestos que aquí claramente no se cumplen (un grupo tiene 1
    municipio y otro 17). Nunca dice "estos datos no se agrupan bien"; siempre entrega una partición.

11. **Los resultados dependen por completo de las variables elegidas.** Con otra agrupación de sectores
    —por ejemplo separando turismo de restaurantes— los grupos podrían cambiar. No hay una única
    agrupación "correcta".

### Sobre las conclusiones

12. **El análisis es descriptivo, no causal.** Dice **qué** municipios se parecen entre sí bajo las
    variables consideradas. **No explica por qué** un municipio tiene ese perfil, ni permite predecir
    qué pasaría si algo cambiara.

13. **El patrón territorial de la gráfica 6 es una observación, no un resultado del modelo.** La
    ubicación geográfica nunca entró a KMeans.

---
## 17. Conclusiones

**La pregunta de investigación era:** *¿Qué municipios del estado de Morelos comparten un perfil
económico similar según la concentración de sus unidades económicas dedicadas a la manufactura, el
comercio y los servicios?*

**Con los datos disponibles y bajo las variables consideradas, los resultados sugieren cuatro perfiles:**

| Grupo | n | Qué lo distingue | Representante |
|---|---|---|---|
| **0 — Especializado en manufactura** | 1 | Manufactura +34.8 pts (confitería) | Temoac |
| **1 — Servicios y cabeceras urbanas** | 17 | Más servicios, unidades más grandes | Jojutla |
| **2 — Abasto y manufactura ligera** | 11 | Comercio al por mayor +1.2 pts | Tetela del Volcán |
| **3 — Consumo final** | 7 | Comercio al por menor y alimentos +3 pts c/u | Tlayacapan |

Leídos en los términos de la pregunta —manufactura, comercio y servicios— los cuatro grupos se ordenan
así: **uno definido por la manufactura** (Temoac), **uno definido por los servicios** (cluster 1),
**uno definido por el comercio de abasto** (cluster 2) y **uno definido por el comercio al menudeo y los
alimentos** (cluster 3). Las tres familias que nombra la pregunta resultaron ser, efectivamente, los ejes
que separan a los municipios.

**Tres hallazgos que vale la pena subrayar:**

1. **El comercio al por menor no distingue a nadie.** Está entre el 28.8% y el 56.2% en los 36
   municipios. Lo que separa a los municipios de Morelos **no es tener comercio, sino qué tienen
   *además* del comercio**.

2. **Hay un caso de especialización genuina.** Temoac se separa de los otros 35 municipios en cualquier
   valor de k que probamos, y la razón es concreta y verificable: 592 establecimientos de elaboración de
   dulces.

3. **Fuera de ese caso, las diferencias son de grado, no de naturaleza.** Los otros tres grupos se
   distinguen por **1 a 3 puntos porcentuales**. Esto tiene una lectura sustantiva: la estructura
   económica de los municipios de Morelos, medida por composición de establecimientos, es **más
   homogénea de lo que uno esperaría** entre una capital estatal y un municipio rural.

**Para qué sirve esto.** La segmentación convierte 113,066 registros sueltos en cuatro perfiles
manejables. Con las reservas del apartado siguiente, sirve como insumo para **focalizar servicios B2B**
(el cluster 1 concentra los municipios con más establecimientos y de mayor tamaño promedio), para
**distribuir soluciones especializadas** según el tipo de actividad dominante, y como punto de partida
para **política pública de desarrollo económico regional** —por ejemplo, un programa de apoyo a la
manufactura tiene un destinatario evidente en Temoac, y uno de fomento al comercio de abasto lo tiene
en el cluster 2.

**Lo que este análisis NO permite concluir.** No sabemos qué municipio "está mejor" ni cuál tiene más
actividad económica: solo describimos **composición del mercado formal**, con establecimientos como
unidad de conteo y una única fotografía en el tiempo. La silhouette baja (0.2074) obliga a tratar estos
grupos como **agrupamientos aproximados y sujetos a revisión**, no como una clasificación definitiva.

### Trabajo futuro

- **Cruzar con población municipal del INEGI** (la clave `cve_geo` ya está lista para eso) para calcular
  unidades económicas per cápita y distinguir tamaño de intensidad económica.
- **Comparar contra otro corte del DENUE** para pasar de una fotografía a una tendencia y ver si los
  grupos son estables en el tiempo.
- **Probar clustering jerárquico como contraste:** al no exigir un k previo y no asumir grupos
  esféricos, permitiría verificar si los cuatro grupos aquí encontrados aparecen también con otro método.

---
## 18. Verificación final de los artefactos generados

In [ ]:
# Comprobamos que el notebook generó todo lo que debía generar.
# Si algo falta, se ve aquí y no cuando ya se entregó.
artefactos = [
    RUTA_PROCESSED / 'dataset_limpio.csv',
    RUTA_OUTPUTS / 'matriz_municipio_sector.csv',
    RUTA_OUTPUTS / 'variables_modelo.csv',
    RUTA_OUTPUTS / 'metricas_k.csv',
    RUTA_OUTPUTS / 'resultados_modelo.csv',
    RUTA_OUTPUTS / 'resumen_resultados.csv',
    RUTA_IMAGENES / 'grafica_1_unidades_por_municipio.png',
    RUTA_IMAGENES / 'grafica_2_composicion_sectorial.png',
    RUTA_IMAGENES / 'grafica_3_seleccion_k.png',
    RUTA_IMAGENES / 'grafica_4_perfil_clusters.png',
    RUTA_IMAGENES / 'grafica_5_pca_clusters.png',
    RUTA_IMAGENES / 'grafica_6_mapa_clusters.png',
]

for ruta in artefactos:
    estado = 'OK ' if ruta.exists() else 'FALTA'
    tam = f'{ruta.stat().st_size / 1024:.1f} KB' if ruta.exists() else '-'
    print(f'[{estado}] {ruta.relative_to(BASE)}  ({tam})')

In [ ]:
# Prueba de reproducibilidad: volvemos a ajustar el modelo con la misma
# semilla y verificamos que las etiquetas sean idénticas.
etiquetas_verificacion = KMeans(
    n_clusters=K_ELEGIDO, random_state=42, n_init=10
).fit_predict(X_scaled)

identico = (etiquetas_verificacion == etiquetas).all()
print('¿Segunda corrida da exactamente el mismo resultado?', bool(identico))
print()
print('Resumen del análisis:')
print('  Registros analizados :', len(df_sel))
print('  Municipios           :', len(df_mun))
print('  Variables del modelo :', len(VARS))
print('  Grupos encontrados   :', K_ELEGIDO)
print('  Silhouette           :', round(silhouette_score(X_scaled, etiquetas), 4))

---
## 19. Guardado del modelo

Cerramos el notebook dejando el modelo listo para volver a usarse.

### ¿Hace falta guardar un modelo no supervisado?

**Para responder la pregunta de investigación, no.** A diferencia de un modelo supervisado —que se
guarda para clasificar registros nuevos conforme van llegando— aquí la población es **cerrada y
completa**: Morelos tiene 36 municipios y **los 36 están en el entrenamiento**. No va a aparecer un
municipio 37 al que haya que asignarle un grupo.

**Aun así lo guardamos, por tres razones concretas:**

1. **Reproducibilidad y auditoría.** Cualquiera puede verificar los grupos sin volver a procesar los
   113,066 registros del DENUE.
2. **Aplicarlo a otra entidad.** Cargar el modelo y ver en qué perfil cae cada municipio de otro estado,
   sin reentrenar. Los grupos quedan definidos por Morelos y sirven como marco de comparación.
3. **Aplicarlo a otro corte temporal.** Comparar el DENUE 2026/06 contra un corte posterior y detectar
   si algún municipio cambió de perfil. Es lo que convierte una de las líneas de trabajo futuro en algo
   ejecutable.

### Objetos guardados

| Objeto | Por qué es indispensable |
|---|---|
| `modelo_kmeans` | El modelo ajustado |
| `escalador` | Fue entrenado con las medias y desviaciones de Morelos. Sin él se le darían datos sin escalar a un modelo entrenado con datos escalados no da error, simplemente devuelve grupos equivocados |
| `VARS` | Las 9 variables deben entrar en el mismo orden si se cambia, cada valor se interpreta como otra variable |
| `mapa_clusters` | KMeans devuelve sus etiquetas originales no las que reordenamos por `pct_com_menor`. Sin este mapeo, el "cluster 2" del modelo cargado no sería el "cluster 2" del artículo |

### Cómo se reutiliza

El orden de la cadena es obligatorio y siempre el mismo:

`datos nuevos → seleccionar las 9 columnas en orden → escalador.transform() → kmeans.predict() → aplicar mapa_clusters`

> Detalle importante: se usa `escalador.transform()`, nunca `fit_transform()`. `fit` volvería a
> calcular medias y desviaciones con los datos nuevos, y entonces los grupos ya no serían comparables
> con los de este análisis.

**Advertencia de alcance:** los grupos quedaron definidos por la estructura económica de Morelos.
Aplicarlos a otro estado responde *"¿a cuál de los perfiles morelenses se parece más este municipio?"*
y no "¿cuáles son los perfiles de ese estado?". Para lo segundo habría que entrenar un modelo nuevo.

La celda final guarda los cuatro objetos y comprueba en el momento que se pueden recuperar
los vuelve a cargar desde disco y reproduce la asignación de los 36 municipios. Guardar un modelo sin
verificar que se puede volver a usar no sirve de nada.

In [6]:
# ---------------------------------------------------------------------
# CELDA FINAL: guardado del modelo y de todo lo necesario para reusarlo
# ---------------------------------------------------------------------
# joblib es la forma estándar de serializar objetos de scikit-learn
import joblib

RUTA_MODELO = RUTA_OUTPUTS / 'modelo'
RUTA_MODELO.mkdir(parents=True, exist_ok=True)

# --- 1. Guardar los cuatro objetos de la cadena ---
objetos = {
    # El modelo ajustado
    'modelo_kmeans.pkl':   modelo_kmeans,
    # El escalador YA ENTRENADO: guarda las medias y desviaciones de Morelos.
    # Sin él, cualquier dato nuevo entraría en una escala distinta.
    'escalador.pkl':       escalador,
    # Las 9 variables EN ORDEN: la columna 0 del array escalado es
    # pct_com_menor, la 1 es pct_com_mayor, etc.
    'columnas_modelo.pkl': VARS,
    # El reetiquetado de la sección 13. Sin esto, el modelo cargado
    # devolvería la numeración arbitraria original de KMeans.
    'mapa_clusters.pkl':   mapa_clusters,
}

for nombre, objeto in objetos.items():
    joblib.dump(objeto, RUTA_MODELO / nombre)
    ruta = RUTA_MODELO / nombre
    print(f'guardado  {nombre:22s} ({ruta.stat().st_size / 1024:.1f} KB)')

# --- 2. Comprobar que se puede recuperar ---
# Cargamos como si fuéramos otra persona en otra computadora
kmeans_cargado    = joblib.load(RUTA_MODELO / 'modelo_kmeans.pkl')
escalador_cargado = joblib.load(RUTA_MODELO / 'escalador.pkl')
columnas_cargadas = joblib.load(RUTA_MODELO / 'columnas_modelo.pkl')
mapa_cargado      = joblib.load(RUTA_MODELO / 'mapa_clusters.pkl')

print()
print('recargado desde disco:', type(kmeans_cargado).__name__,
      '| k =', kmeans_cargado.n_clusters,
      '|', len(columnas_cargadas), 'columnas |', 'mapeo', mapa_cargado)

# reindex asegura columnas completas y EN EL ORDEN esperado.
# OJO: aquí NO usamos fill_value=0 como en el modelo supervisado. Allá las
# columnas eran variables dummy y un 0 significaba "esa categoría no está".
# Aquí un 0 sería un PORCENTAJE REAL (0% de manufactura) y falsearía el perfil
# en silencio. Preferimos que falte y que el assert lo detecte.
datos_nuevos = df_mun.reindex(columns=columnas_cargadas)
assert datos_nuevos.isnull().sum().sum() == 0, 'Faltan columnas que el modelo necesita'

# transform (NO fit_transform): reutiliza las medias y desviaciones de Morelos
datos_escalados = escalador_cargado.transform(datos_nuevos)

# predict + reetiquetado, en ese orden
prediccion = pd.Series(kmeans_cargado.predict(datos_escalados),
                       index=df_mun.index).map(mapa_cargado)

coinciden = (prediccion == df_mun['cluster']).sum()
print()
print('Municipios que reciben el mismo grupo:', coinciden, 'de', len(df_mun))

if coinciden == len(df_mun):
    print('OK: el modelo guardado reproduce el resultado del análisis.')
    print('    Los cuatro archivos quedaron en outputs/modelo/')
else:
    print('ATENCIÓN: hay diferencias. Revisar la cadena de guardado.')
    print(df_mun.loc[prediccion != df_mun['cluster'], ['cluster']])

NameError: name 'modelo_kmeans' is not defined